# 16. Conditioning and DiT modulation — FiLM, cross-attention, DiT-B/2, SD3 MMDiT

Only widths and token/image counts are reduced. Both DiT and SD3 use fixed 2D sin-cos patch positional embeddings rather than learnable replacements.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")


## 1. Conditioning primitives


In [ ]:
features = torch.randn(2, 5, 8)
condition = torch.randn(2, 4)
film = nn.Linear(4, 16)
scale, shift = film(condition).chunk(2, -1)
film_output = features * (1 + scale[:, None]) + shift[:, None]
assert film_output.shape == features.shape

image_tokens = torch.randn(2, 6, 12)
text_tokens = torch.randn(2, 4, 12)
q = nn.Linear(12, 12, bias=False)(image_tokens).view(2, 6, 3, 4).transpose(1, 2)
k = nn.Linear(12, 12, bias=False)(text_tokens).view(2, 4, 3, 4).transpose(1, 2)
v = nn.Linear(12, 12, bias=False)(text_tokens).view(2, 4, 3, 4).transpose(1, 2)
assert F.scaled_dot_product_attention(q, k, v).shape == (2, 3, 6, 4)


## 2. Fixed 2D sin-cos helper


In [ ]:
def sincos_1d(dim, positions):
    assert dim % 2 == 0
    omega = torch.arange(dim // 2, device=positions.device, dtype=torch.float32)
    omega = 1.0 / (10000 ** (omega / (dim / 2)))
    angles = positions.reshape(-1, 1).float() * omega.reshape(1, -1)
    return torch.cat([angles.sin(), angles.cos()], -1)


def sincos_2d(dim, grid_size, device):
    assert dim % 4 == 0
    y, x = torch.meshgrid(
        torch.arange(grid_size, device=device),
        torch.arange(grid_size, device=device),
        indexing="ij",
    )
    return torch.cat(
        [
            sincos_1d(dim // 2, x.reshape(-1)),
            sincos_1d(dim // 2, y.reshape(-1)),
        ],
        -1,
    )[None]


def timestep_embedding(t, dim):
    half = dim // 2
    freq = torch.exp(
        -math.log(10000.0)
        * torch.arange(half, device=t.device, dtype=torch.float32)
        / half
    )
    args = t.float()[:, None] * freq[None]
    return torch.cat([args.cos(), args.sin()], -1)


def modulate(x, shift, scale):
    return x * (1 + scale[:, None]) + shift[:, None]


## 3. DiT-B/2 — 12 blocks, 12 heads, adaLN-Zero, fixed 2D sin-cos PE


In [ ]:
class DiTAttention(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):
        batch, length, dim = x.shape
        qkv = self.qkv(x).view(batch, length, 3, self.heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        y = F.scaled_dot_product_attention(q, k, v)
        return self.out(y.transpose(1, 2).contiguous().view(batch, length, dim))


class DiTBlock(nn.Module):
    def __init__(self, dim=48, heads=12):
        super().__init__()
        self.attn = DiTAttention(dim, heads)
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * dim, dim),
        )
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.ada[-1].weight)
        nn.init.zeros_(self.ada[-1].bias)

    def forward(self, x, c):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = self.ada(c).chunk(6, -1)
        x = x + gate_a[:, None] * self.attn(modulate(self.norm1(x), shift_a, scale_a))
        return x + gate_m[:, None] * self.mlp(modulate(self.norm2(x), shift_m, scale_m))


class DiTFinal(nn.Module):
    def __init__(self, dim, patch_size, out_channels):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        self.out = nn.Linear(dim, patch_size * patch_size * out_channels)
        nn.init.zeros_(self.ada[-1].weight)
        nn.init.zeros_(self.ada[-1].bias)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, x, c):
        shift, scale = self.ada(c).chunk(2, -1)
        return self.out(modulate(self.norm(x), shift, scale))


class SmallWidthDiTB2(nn.Module):
    def __init__(self, image_size=8, channels=4, dim=48, depth=12, heads=12, classes=10):
        super().__init__()
        self.image_size = image_size
        self.patch = nn.Conv2d(channels, dim, 2, stride=2)
        position = sincos_2d(
            dim,
            image_size // 2,
            torch.device("cpu"),
        )
        self.register_buffer("position", position, persistent=False)
        self.time = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.label = nn.Embedding(classes + 1, dim)
        self.blocks = nn.ModuleList([DiTBlock(dim, heads) for _ in range(depth)])
        self.final = DiTFinal(dim, 2, channels * 2)

    def forward(self, image, t, label):
        x = self.patch(image).flatten(2).transpose(1, 2)
        x = x + self.position.to(x.device, x.dtype)
        c = self.time(timestep_embedding(t, x.size(-1))) + self.label(label)
        for block in self.blocks:
            x = block(x, c)
        patches = self.final(x, c).transpose(1, 2)
        return F.fold(patches, (self.image_size, self.image_size), kernel_size=2, stride=2)


dit = SmallWidthDiTB2()
assert len(dit.blocks) == 12
assert all(block.attn.heads == 12 for block in dit.blocks)
assert not isinstance(dit.position, nn.Parameter)
dit_output = dit(
    torch.randn(1, 4, 8, 8),
    torch.tensor([500]),
    torch.tensor([2]),
)
dit_output.square().mean().backward()


## 4. Stable Diffusion 3 Medium MMDiT — 24 joint blocks / 24 heads / final context-pre-only block


In [ ]:
class AdaLNZero(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))

    def forward(self, x, c):
        shift_a, scale_a, gate_a, shift_m, scale_m, gate_m = self.modulation(c).chunk(6, -1)
        return modulate(self.norm(x), shift_a, scale_a), gate_a, shift_m, scale_m, gate_m


class AdaLNContinuous(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))

    def forward(self, x, c):
        shift, scale = self.modulation(c).chunk(2, -1)
        return modulate(self.norm(x), shift, scale)


class Stream(nn.Module):
    def __init__(self, dim=48, heads=24, pre_only=False):
        super().__init__()
        self.pre_only = pre_only
        self.heads = heads
        self.head_dim = dim // heads
        self.norm1 = AdaLNContinuous(dim) if pre_only else AdaLNZero(dim)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        if not pre_only:
            self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
            self.mlp = nn.Sequential(
                nn.Linear(dim, 4 * dim),
                nn.GELU(approximate="tanh"),
                nn.Linear(4 * dim, dim),
            )

    def qkv_project(self, x):
        batch, length, dim = x.shape
        qkv = self.qkv(x).view(batch, length, 3, self.heads, self.head_dim)
        return qkv.permute(2, 0, 3, 1, 4).unbind(0)


class JointBlock(nn.Module):
    def __init__(self, dim=48, heads=24, context_pre_only=False):
        super().__init__()
        self.context_pre_only = context_pre_only
        self.image = Stream(dim, heads, pre_only=False)
        self.context = Stream(dim, heads, pre_only=context_pre_only)

    @staticmethod
    def merge(x):
        return x.transpose(1, 2).contiguous().flatten(2)

    def forward(self, image, context, c):
        image_n, image_gate, image_shift, image_scale, image_mlp_gate = self.image.norm1(image, c)
        if self.context_pre_only:
            context_n = self.context.norm1(context, c)
        else:
            (
                context_n,
                context_gate,
                context_shift,
                context_scale,
                context_mlp_gate,
            ) = self.context.norm1(context, c)

        iq, ik, iv = self.image.qkv_project(image_n)
        cq, ck, cv = self.context.qkv_project(context_n)
        q = torch.cat([iq, cq], dim=2)
        k = torch.cat([ik, ck], dim=2)
        v = torch.cat([iv, cv], dim=2)
        joint = F.scaled_dot_product_attention(q, k, v)

        image_len = image.size(1)
        image_attn = self.merge(joint[:, :, :image_len])
        context_attn = self.merge(joint[:, :, image_len:])
        image = image + image_gate[:, None] * self.image.out(image_attn)
        image_mlp_in = modulate(self.image.norm2(image), image_shift, image_scale)
        image = image + image_mlp_gate[:, None] * self.image.mlp(image_mlp_in)

        if not self.context_pre_only:
            context = context + context_gate[:, None] * self.context.out(context_attn)
            context_mlp_in = modulate(self.context.norm2(context), context_shift, context_scale)
            context = context + context_mlp_gate[:, None] * self.context.mlp(context_mlp_in)
        return image, context


class SmallWidthSD3Medium(nn.Module):
    def __init__(self, sample_size=8, in_channels=16, dim=48, depth=24, heads=24, context_dim=32):
        super().__init__()
        self.patch = nn.Conv2d(in_channels, dim, 2, stride=2)
        grid = sample_size // 2
        self.register_buffer("position", sincos_2d(dim, grid, torch.device("cpu")), persistent=False)
        self.context_projection = nn.Linear(context_dim, dim)
        self.time = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.blocks = nn.ModuleList(
            [
                JointBlock(dim, heads, context_pre_only=(index == depth - 1))
                for index in range(depth)
            ]
        )

    def forward(self, image, context, t):
        x = self.patch(image).flatten(2).transpose(1, 2)
        x = x + self.position.to(x.device, x.dtype)
        context = self.context_projection(context)
        c = self.time(timestep_embedding(t, x.size(-1)))
        for block in self.blocks:
            x, context = block(x, context, c)
        return x


sd3 = SmallWidthSD3Medium()
assert len(sd3.blocks) == 24
assert all(block.image.heads == 24 for block in sd3.blocks)
assert sd3.blocks[-1].context_pre_only
assert not isinstance(sd3.position, nn.Parameter)
sd3_output = sd3(
    torch.randn(1, 16, 8, 8),
    torch.randn(1, 5, 32),
    torch.tensor([500]),
)
sd3_output.square().mean().backward()


## Audit result

The learnable patch-position substitutes are removed from both DiT and SD3. The fixed 2D sin-cos positional path is explicit and asserted.
